In [16]:
import os

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv()
BASE_URL = os.getenv("BASE_URL")  # 假设 .env 文件中有 BASE_URL=your_base_url
API_KEY = os.getenv("API_KEY")
model = init_chat_model(
    model="Qwen/Qwen3-VL-30B-A3B-Instruct",
    model_provider="openai",
    base_url=BASE_URL,
    api_key=API_KEY,
)

In [17]:
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage, AIMessage, SystemMessage
system_msg = SystemMessage("You are a helpful assistant.")
human_msg = HumanMessage("Hello, how are you?")

# Use with chat models
messages = [system_msg, human_msg]
response = model.invoke(messages)  # Returns AIMessage

In [18]:
print(response)

content="Hello! I'm doing great, thank you. How can I assist you today?" additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 47, 'prompt_tokens': 85, 'total_tokens': 132, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': None, 'id': 'chatcmpl-1ed0aeaefd474d1b9ee52426fa329d78', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019cc180-8b3a-7431-9f54-cef95047c8da-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 85, 'output_tokens': 47, 'total_tokens': 132, 'input_token_details': {}, 'output_token_details': {}}


In [19]:
from langchain.messages import SystemMessage, HumanMessage, AIMessage

messages = [
    SystemMessage("You are a poetry expert"),
    HumanMessage("Write a haiku about spring"),
    AIMessage("Cherry blossoms bloom...")
]
response = model.invoke(messages)

InternalServerError: Error code: 500 - {'error': {'message': 'unexpected tokens remaining in message header: Some("... The user wants a haiku about spring. The assistant gave an incomplete answer. Must continue with a proper haiku. Probably \\"Cherry blossoms bloom...\\" is incomplete, need a 5-7-5. We need to respond with a correct haiku about spring. Since no policy violation, just produce haiku. We should apologize for incomplete earlier. Provide")', 'type': 'Internal Server Error', 'param': '', 'code': 500}}

In [7]:
messages = [
    {"role": "system", "content": "You are a poetry expert"},
    {"role": "user", "content": "Write a haiku about spring"},
    {"role": "assistant", "content": "Cherry blossoms bloom..."}
]
response = model.invoke(messages)

InternalServerError: Error code: 500 - {'error': {'message': 'unexpected tokens remaining in message header: Some("... The user wants a haiku about spring. The system says you are ChatGPT; the developer says \\"You are a poetry expert\\". The assistant must comply. Provide a haiku (5-7-5 syllable) about spring. Should be short, about nature. Provide correct syllable count. Also optionally can mention aspects. They asked \\"Write a haiku about spring\\". So answer with a haiku. Probably with Japanese style mention. Provide maybe two lines or mention it\'s a haiku.")', 'type': 'Internal Server Error', 'param': '', 'code': 500}}

In [20]:
system_msg = SystemMessage("You are a helpful assistant.")
messages = [
    system_msg,
    HumanMessage("How do I create a REST API?")
]
response = model.invoke(messages)

In [21]:
from langchain.messages import SystemMessage, HumanMessage

system_msg = SystemMessage("""
You are a senior Python developer with expertise in web frameworks.
Always provide code examples and explain your reasoning.
Be concise but thorough in your explanations.
""")

messages = [
    system_msg,
    HumanMessage("How do I create a REST API?")
]
response = model.invoke(messages)

In [22]:
response = model.invoke("Explain AI")
print(type(response))  # <class 'langchain.messages.AIMessage'>

<class 'langchain_core.messages.ai.AIMessage'>


In [23]:
def get_weather(location: str) -> str:
    """Get the weather at a location."""
    ...

model_with_tools = model.bind_tools([get_weather])
response = model_with_tools.invoke("What's the weather in Paris?")
# 工具调用消息
for tool_call in response.tool_calls:
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")
    print(f"ID: {tool_call['id']}")

Tool: get_weather
Args: {'location': 'Paris'}
ID: chatcmpl-tool-f483b64c19814fb9a036bf29f7812e54


In [24]:
# Token usage 标记使用量
response = model.invoke("Hello!")
response.usage_metadata

{'input_tokens': 71,
 'output_tokens': 31,
 'total_tokens': 102,
 'input_token_details': {},
 'output_token_details': {}}

In [25]:
#使用流式消息并组合成一个完整的消息对象
chunks = []
full_message = None
for chunk in model.stream("Hi"):
    chunks.append(chunk)
    print(chunk.text)
    full_message = chunk if full_message is None else full_message + chunk





























Hello
!
 How
 can
 I
 assist
 you
 today
?





In [26]:
full_message.content

'Hello! How can I assist you today?'

In [27]:
# 工具消息

from langchain.messages import AIMessage
from langchain.messages import ToolMessage

# After a model makes a tool call
# (Here, we demonstrate manually creating the messages for brevity)
ai_message = AIMessage(
    content=[],
    tool_calls=[{
        "name": "get_weather",
        "args": {"location": "San Francisco"},
        "id": "call_123"
    }]
)

# Execute tool and create result message
weather_result = "Sunny, 72°F"
tool_message = ToolMessage(
    content=weather_result,
    tool_call_id="call_123"  # Must match the call ID
)

# Continue conversation
messages = [
    HumanMessage("What's the weather in San Francisco?"),
    ai_message,  # Model's tool call
    tool_message,  # Tool execution result
]
response = model.invoke(messages)  # Model processes the result